# Lesson 01 Lab — CMOS Switching, State, and Dynamic Power

**Puzzle:** A transistor stores no Python value, so how can billions of switches implement state—and why does voltage dominate switching energy?

This notebook retains one complete RTX 5090 execution.


## Why this matters

GPU performance begins with a physical transition. A CMOS inverter maps an input voltage to one of two stable output regions; cross-coupled inverters can then hold a bit. The useful systems connection is not transistor trivia. Every clocked transition charges or discharges capacitance, so activity, voltage, capacitance, and frequency set a first-order power envelope long before CUDA exposes a kernel.


## 0. Predict before running

1. Predict the inverter output for low and high inputs.
2. Predict the energy ratio between 1.0 V and 0.8 V at fixed capacitance.
3. Name two power terms that the dynamic model omits.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

For one effective capacitance, a 0→1 transition draws roughly `C·V²` from the supply; about half is stored and the rest is dissipated, then the stored energy is dissipated on discharge. A common activity-averaged model is `P_dynamic ≈ α·C·V²·f`. It is a model, not a board-power meter: leakage, short-circuit current, clock trees, memories, regulators, and workload placement add terms. The experiment keeps the equation explicit and sweeps one variable at a time so the quadratic voltage dependence cannot be confused with a measured GPU wattage claim.

- Logic state is represented by voltage ranges, not by a software type.
- Cross-coupled feedback creates state; an isolated inverter only transforms a signal.
- The `V²` term makes voltage changes more consequential than equal percentage frequency changes in this model.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["input voltage"] --> B["PMOS/NMOS conduction"]
  B --> C["output capacitance charges or discharges"]
  C --> D["logic state"]
  C --> E["dynamic energy ≈ C·V²"]
```


## 3. Inspect the visual boundary

![CMOS inverter states](../assets/visualizations/cmos-inverter.png)

- [Interactive inverter visualization](../assets/visualizations/cmos-inverter.html)

These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 1
LESSON_TITLE = 'CMOS Switching, State, and Dynamic Power'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260814
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | 1.0 V, 1 GHz, fixed effective capacitance and activity |
| Candidate | 0.8 V and changed activity/frequency scenarios |
| Held constant | capacitance, activity, and the selected sweep variable |
| Measurements | energy per transition, dynamic power, and voltage energy ratio |
| Evidence | `numerical-model` |

**Experiment:** Evaluate the inverter truth table and sweep the transparent `αCV²f` model.


## 6. Inspect the code

The code expresses the equations directly in SI units, then reports femtojoules and milliwatts for readable scales. It does not query board power or infer a voltage-frequency curve from the GPU.

Do not run until the code matches the frozen table.


In [2]:
C = 80e-15
alpha = 0.18
frequency = 1e9

def transition_energy(capacitance_f, voltage_v):
    return capacitance_f * voltage_v**2

def dynamic_power(capacitance_f, voltage_v, activity, frequency_hz):
    return activity * capacitance_f * voltage_v**2 * frequency_hz

truth_table = {0: 1, 1: 0}
e_1v = transition_energy(C, 1.0)
e_08v = transition_energy(C, 0.8)
voltage_sweep = {
    str(v): dynamic_power(C, v, alpha, frequency) * 1e3
    for v in (0.6, 0.7, 0.8, 0.9, 1.0)
}
activity_sweep = {
    str(a): dynamic_power(C, 1.0, a, frequency) * 1e3
    for a in (0.05, 0.1, 0.18, 0.3, 0.5)
}
metrics = {
    "truth_table": truth_table,
    "capacitance_f": C,
    "activity": alpha,
    "frequency_hz": frequency,
    "energy_1v_fj": e_1v * 1e15,
    "energy_08v_fj": e_08v * 1e15,
    "voltage_energy_ratio": e_1v / e_08v,
    "power_1v_mw": dynamic_power(C, 1.0, alpha, frequency) * 1e3,
    "voltage_sweep_mw": voltage_sweep,
    "activity_sweep_mw": activity_sweep,
}
analysis = (
    f"At fixed C, activity, and frequency, lowering voltage from 1.0 V to 0.8 V "
    f"reduced modeled transition energy from {metrics['energy_1v_fj']:.1f} to "
    f"{metrics['energy_08v_fj']:.1f} fJ, a {metrics['voltage_energy_ratio']:.3f}x ratio. "
    "This is a sensitivity model, not board-power telemetry."
)
print(json.dumps(metrics, indent=2))


{
  "truth_table": {
    "0": 1,
    "1": 0
  },
  "capacitance_f": 8e-14,
  "activity": 0.18,
  "frequency_hz": 1000000000.0,
  "energy_1v_fj": 80.0,
  "energy_08v_fj": 51.20000000000001,
  "voltage_energy_ratio": 1.5624999999999996,
  "power_1v_mw": 0.014400000000000001,
  "voltage_sweep_mw": {
    "0.6": 0.005184,
    "0.7": 0.007056,
    "0.8": 0.009216000000000002,
    "0.9": 0.011664000000000002,
    "1.0": 0.014400000000000001
  },
  "activity_sweep_mw": {
    "0.05": 0.004000000000000001,
    "0.1": 0.008000000000000002,
    "0.18": 0.014400000000000001,
    "0.3": 0.023999999999999997,
    "0.5": 0.04
  }
}


## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Energy at 1.0 V | 80.0000 |
| Energy at 0.8 V | 51.2000 |
| Energy ratio | 1.562x |
| Baseline dynamic power | 0.0144 |


## 8. Explain rather than overclaim

At fixed C, activity, and frequency, lowering voltage from 1.0 V to 0.8 V reduced modeled transition energy from 80.0 to 51.2 fJ, a 1.562x ratio. This is a sensitivity model, not board-power telemetry.

**Evidence boundary:** A transparent mechanism model executed. It establishes the stated relationship under printed assumptions, not native hardware latency, energy, or topology.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 1, "title": 'CMOS Switching, State, and Dynamic Power', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Use `αCV²f` to reason about direction and sensitivity; use hardware telemetry and controlled workloads to measure an actual GPU.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 1,
  "title": "CMOS Switching, State, and Dynamic Power",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260814
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "truth_table": {
      "0": 1,
      "1": 0
    },
    "capacitance_f": 8e-14,
    "activity": 0.18,
    "frequency_hz": 1000000000.0,
    "energy_1v_fj": 80.0,
    "energy_08v_fj": 51.20000000000001,
    "voltage_energy_ratio": 1.5624999999999996,
    "power_1v_mw": 0.014400000000000001,
    "voltage_sweep_mw": {
      "0.6": 0.005184,
      "0.7": 0.007056,
      "0.8": 0.009216000000000002,
      "0.9": 0.011664000000000002,
      "1.0": 0.014400000000000001
    },
    "activity_sweep_mw": {
      "0.05": 0.004000000000000001,
      "0.1": 0.008000000000000002,
      "0.18": 0.014400000000000001,
      "0.3": 0.023999999999999997,
      "0.5": 0.04
    }
  },
 

## 10. Make the decision

> Use `αCV²f` to reason about direction and sensitivity; use hardware telemetry and controlled workloads to measure an actual GPU.

**Failure analysis:** The effective capacitance and voltage are illustrative. Real dynamic voltage/frequency scaling changes several variables together, and leakage can become important at different process and temperature points.


## 11. Extend the evidence

Collect board-power samples for a fixed CUDA workload at several locked clocks, then compare measured deltas with the model's direction rather than forcing an exact fit.

See [`README.md`](README.md) for the full explanation and references.
